# Experiment 5.0 — Local-evidence objective × event-capacity analysis

Analysis-only notebook. Training/evaluation is performed by the Slurm pipeline.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks/artifacts/experiment_5_0_local_evidence_objectives/local_objective_x_event_capacity_v1'
RUNS = pd.read_csv(ROOT / 'runs.csv')
SUMMARY = pd.read_csv(ROOT / 'summary.csv')
LOSS_EFFECTS = pd.read_csv(ROOT / 'paired_loss_effects_summary.csv')
VARIANT_EFFECTS = pd.read_csv(ROOT / 'paired_variant_effects_summary.csv')
MANIFEST = json.loads((ROOT / 'manifest.json').read_text(encoding='utf-8'))
MANIFEST


## Primary deployment result — Output WholeCount

All training objectives are compared using the same final deployment readout.


In [ ]:
test = RUNS[RUNS['split'] == 'test'].copy()
primary = test.groupby(['objective','variant'])['output_whole_count_ba'].agg(['mean','std']).reset_index()
display(primary.sort_values('mean', ascending=False))
pivot = primary.pivot(index='objective', columns='variant', values='mean')
ax = pivot.plot(kind='bar', figsize=(11,5))
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp5.0: Output WholeCount BA')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## Hidden local-evidence quality — L2 WholeCount + Linear


In [ ]:
hidden = test.groupby(['objective','variant'])['l2_whole_count_linear_ba'].agg(['mean','std']).reset_index()
display(hidden.sort_values('mean', ascending=False))
pivot = hidden.pivot(index='objective', columns='variant', values='mean')
ax = pivot.plot(kind='bar', figsize=(11,5))
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp5.0: L2 WholeCount + Linear')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## Readout compression matrix

Compare phase-aware external Linear probes, orderless hidden WholeCount + Linear, endpoint membrane, and the 12-neuron Output WholeCount.


In [ ]:
readout_cols = {
    'Output WholeCount': 'output_whole_count_ba',
    'L1 Count + Linear': 'l1_whole_count_linear_ba',
    'L2 Count + Linear': 'l2_whole_count_linear_ba',
    'L2 Fixed250 + Linear': 'l2_fixed250_ordered_linear_ba',
    'L2 Relative10 + Linear': 'l2_relative10_ordered_linear_ba',
    'L2 Uend + Linear': 'l2_uend_linear_ba',
}
matrix_rows = []
for (objective, variant), part in test.groupby(['objective','variant']):
    row = {'objective': objective, 'variant': variant}
    for label, column in readout_cols.items():
        row[label] = part[column].mean()
    matrix_rows.append(row)
READOUT_MATRIX = pd.DataFrame(matrix_rows)
display(READOUT_MATRIX)


## Paired objective effects vs WholeCount CE


In [ ]:
display(LOSS_EFFECTS.sort_values(['readout','variant','mean'], ascending=[True,True,False]))


## Paired event-capacity effects


In [ ]:
display(VARIANT_EFFECTS.sort_values(['readout','objective','mean'], ascending=[True,True,False]))


## Event cost and persistence


In [ ]:
cost_cols = [
    'l1_events_per_neuron_second',
    'l2_events_per_neuron_second',
    'output_events_per_neuron_second',
    'l1_tail_event_fraction',
    'l2_tail_event_fraction',
    'output_tail_event_fraction',
]
display(test.groupby(['objective','variant'])[cost_cols].mean().reset_index())
cost = test.groupby(['objective','variant']).agg(
    ba=('output_whole_count_ba','mean'),
    output_rate=('output_events_per_neuron_second','mean'),
).reset_index()
fig, ax = plt.subplots(figsize=(8,5))
for variant, part in cost.groupby('variant'):
    ax.scatter(part.output_rate, part.ba, label=variant)
    for row in part.itertuples(index=False):
        ax.annotate(row.objective, (row.output_rate, row.ba), fontsize=8)
ax.set_xlabel('Output events / neuron / second')
ax.set_ylabel('Test Output WholeCount BA')
ax.set_title('Accuracy vs output event cost')
ax.legend()
plt.tight_layout()
plt.show()
